In [1]:
import numpy as np
import pandas as pd
import yfinance as yf


ticker = "CL=F"
start  = "2015-01-01"
end    = None   # up to today

df = yf.download(ticker, start=start, end=end, progress=False)
df = df[["Close"]].dropna()
df.rename(columns={"Close": "price"}, inplace=True)


# Log-returns

df["log_ret"] = np.log(df["price"] / df["price"].shift(1))
df = df.dropna()

# Time step (daily)
DT = 1.0 / 252.0


# Robust diffusion volatility estimate
# (winsorized to reduce jump contamination)
def winsorized_std(x, lower=0.01, upper=0.99):
    lo, hi = np.quantile(x, [lower, upper])
    x_w = np.clip(x, lo, hi)
    return x_w.std(ddof=1)

sigma_diff = winsorized_std(df["log_ret"].values) / np.sqrt(DT)


# Jump detection via threshold rule

C = 4.0  # threshold multiplier (3–5 is standard)
threshold = C * sigma_diff * np.sqrt(DT)

df["is_jump"] = np.abs(df["log_ret"]) > threshold


# Estimate lambda (jump intensity)

n_jumps = df["is_jump"].sum()
n_days  = len(df)
T_years = n_days / 252.0

lambda_jump = n_jumps / T_years


# Estimate jump size distribution
#    Y ≈ log-return on jump days

Y = df.loc[df["is_jump"], "log_ret"].values

mu_J    = Y.mean()
sigma_J = Y.std(ddof=1)


# Jump compensator

kappa = np.exp(mu_J + 0.5 * sigma_J**2) - 1.0


# Results

print("=== Jump-Diffusion Parameter Estimates (WTI) ===")
print(f"Sample period: {df.index.min().date()} → {df.index.max().date()}")
print(f"Observations: {n_days}")
print(f"Detected jumps: {n_jumps}")
print("")
print(f"lambda_jump  = {lambda_jump:.3f}  (jumps / year)")
print(f"mu_J         = {mu_J:.4f}")
print(f"sigma_J      = {sigma_J:.4f}")
print(f"kappa        = {kappa:.4f}")


YF.download() has changed argument auto_adjust default to True
=== Jump-Diffusion Parameter Estimates (WTI) ===
Sample period: 2015-01-05 → 2026-01-02
Observations: 2764
Detected jumps: 24

lambda_jump  = 2.188  (jumps / year)
mu_J         = 0.0196
sigma_J      = 0.1817
kappa        = 0.0368


/opt/homebrew/lib/python3.11/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


In [5]:
import plotly.graph_objects as go

fig_ret = go.Figure()

# Log-returns
fig_ret.add_trace(
    go.Scatter(
        x=df.index,
        y=df["log_ret"],
        mode="lines",
        name="Log-returns"
    )
)

# Jump points
fig_ret.add_trace(
    go.Scatter(
        x=df.index[df["is_jump"]],
        y=df.loc[df["is_jump"], "log_ret"],
        mode="markers",
        name="Detected jumps",
        marker=dict(size=6)
    )
)

# Threshold lines
fig_ret.add_hline(
    y=threshold,
    line_dash="dash",
    annotation_text="+ threshold"
)
fig_ret.add_hline(
    y=-threshold,
    line_dash="dash",
    annotation_text="- threshold"
)

fig_ret.update_layout(
    title="WTI Log-Returns with Jump Detection",
    xaxis_title="Date",
    yaxis_title="Log-return",
    template="plotly_white",
    height=400
)

fig_ret.show()
